### Базисный параллакс Wolf359 

In [1]:
from astropy.io import fits
from math import *
import numpy as np

import assist
ephem = assist.Ephem("/usr/local/etc/de/linux_p1550p2650.440", "/usr/local/etc/de/sb441-n16.bsp")

from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.time import TimeDelta

import astropy.constants as const

def angular_distance(ra,dec,RA,DEC):
    return acos(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))

In [2]:
c_gb = SkyCoord('10:56:23.636 +06:59:57.970', unit=(u.hourangle, u.deg))
c_sp = SkyCoord('10:56:22.641 +07:00:03.673', unit=(u.hourangle, u.deg))
plx = degrees(angular_distance(c_gb.ra.radian,c_gb.dec.radian,c_sp.ra.radian,c_sp.dec.radian))*3600
print('plx = %7.3f arcsec'%plx)

plx =  15.874 arcsec


In [3]:
# координаты КА в момент наблюдений относительно Солнца
hdul = fits.open('fits/lor_0449933827_0x633_sci.fits')
X = hdul[0].header['SPCSSCX']*u.km.to('au')
Y = hdul[0].header['SPCSSCY']*u.km.to('au')
Z = hdul[0].header['SPCSSCZ']*u.km.to('au')

t = Time(hdul[0].header['SPCUTCAL'])+TimeDelta(float(hdul[0].header['EXPTIME']) / 2.0, format='sec')
JD = float(t.copy(format='jd').value)
JD

2458962.8230277756

In [4]:
#координаты Земли и Солнца относительно барицентра
x_e,y_e,z_e = ephem.get_particle("earth", JD-ephem.jd_ref).xyz
x_s,y_s,z_s = ephem.get_particle("sun", JD-ephem.jd_ref).xyz
x_es,y_es,z_es = x_e - x_s, y_e - y_s, z_e - z_s

In [5]:
# координаты КА относительно Земли
X_sc,Y_sc,Z_sc = X - x_es,Y - y_es,Z - z_es
R = sqrt(X_sc*X_sc+Y_sc*Y_sc+Z_sc*Z_sc)
# единичный вектор, дающий направление c КА на Землю
r = -np.array([X_sc/R,Y_sc/R,Z_sc/R])
# единичный вектор, дающий направление на Wolf359 с КА
sc = np.array([cos(c_sp.ra.radian)*cos(c_sp.dec.radian),
              sin(c_sp.ra.radian)*cos(c_sp.dec.radian),
              sin(c_sp.dec.radian)])
sinTheta = np.linalg.norm(np.cross(r,sc))
D = 206265*R*sinTheta/plx
D/206265 

2.5093068444789974

In [6]:
import ssl
from astroquery.gaia import Gaia
limmag1,limmag2 = 10,14
fov = 0.02
ssl._create_default_https_context = ssl._create_unverified_context
job = Gaia.launch_job_async("SELECT * \
FROM gaiadr3.gaia_source \
WHERE CONTAINS(POINT('ICRS',gaiadr3.gaia_source.ra,gaiadr3.gaia_source.dec),CIRCLE('ICRS',%f,%f,%f))=1\
                           AND  phot_g_mean_mag>%f AND  phot_g_mean_mag<%f;"%(c_gb.ra.degree,c_gb.dec.degree,fov,limmag1,limmag2) \
                            , dump_to_file=False)

gaiat = job.get_results()

INFO: Query finished. [astroquery.utils.tap.core]


In [7]:
1000/gaiat['parallax'][0]

2.408597252748958